# 2.2 Embeddings Visuels

Encodage des images de produits avec deux méthodes :
- **Méthode 1** : ResNet-50 pré-entraîné (baseline)
- **Méthode 2** : Vision Transformer (ViT)

**Pour chaque méthode** : calcul de la matrice de similarité + extraction des top-5 voisins pour chaque produit ancre.

## Imports et Chargement des données

In [1]:
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
import torch
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Charger le dataset
df = pd.read_csv('data/subset_fashion_dataset/products_final.csv')
print(f"Dataset chargé : {len(df)} produits")

# Charger le ground truth
ground_truth = pd.read_csv('data/subset_fashion_dataset/ground_truth.csv', sep=';')
print(f"Ground truth chargé : {len(ground_truth)} ancres")

# Mapping ID produit → index
id_to_index = {row['id']: idx for idx, row in df.iterrows()}
print(f"Mapping créé : {len(id_to_index)} produits")

Dataset chargé : 400 produits
Ground truth chargé : 30 ancres
Mapping créé : 400 produits


In [3]:
# Configurer les chemins d'images
IMAGES_DIR = Path('data/subset_fashion_dataset/images')

# Vérifier que les images existent
sample_img = IMAGES_DIR / df['image_path'].iloc[0].split('/')[-1]
print(f"Image exemple : {sample_img}")
print(f"Existe : {sample_img.exists()}")

Image exemple : data\subset_fashion_dataset\images\15832.jpg
Existe : True


---
## Méthode 1 : ResNet-50 (Baseline)

In [4]:
# Charger ResNet-50 pré-entraîné
resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

# Retirer la dernière couche (classification) pour garder les features
resnet_features = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet_features.eval()

print("ResNet-50 chargé (avant-dernière couche : 2048 dimensions)")

ResNet-50 chargé (avant-dernière couche : 2048 dimensions)


In [5]:
# Prétraitement des images (standard ImageNet)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # Moyenne ImageNet
        std=[0.229, 0.224, 0.225]    # Écart-type ImageNet
    )
])

print("Prétraitement configuré")

Prétraitement configuré


In [6]:
# Extraire les features pour toutes les images
resnet_embeddings = []
missing_images = []

print("Extraction des features ResNet-50 en cours...")
for idx, row in df.iterrows():
    img_name = row['image_path'].split('/')[-1]
    img_path = IMAGES_DIR / img_name
    
    if not img_path.exists():
        missing_images.append(idx)
        resnet_embeddings.append(np.zeros(2048))
        continue
    
    # Charger et prétraiter l'image
    img = Image.open(img_path).convert('RGB')
    img_tensor = preprocess(img).unsqueeze(0)
    
    # Extraire les features
    with torch.no_grad():
        features = resnet_features(img_tensor)
        features = features.flatten().numpy()
    
    resnet_embeddings.append(features)

resnet_embeddings = np.array(resnet_embeddings)
print(f"\nFeatures ResNet-50 : {resnet_embeddings.shape}")
print(f"Images manquantes : {len(missing_images)}")

Extraction des features ResNet-50 en cours...

Features ResNet-50 : (400, 2048)
Images manquantes : 0


In [7]:
# Matrice de similarité cosinus
resnet_similarity = cosine_similarity(resnet_embeddings)
print(f"Matrice de similarité : {resnet_similarity.shape}")

Matrice de similarité : (400, 400)


In [8]:
# Extraction top-5 pour les ancres
def get_top5_neighbors(anchor_id, similarity_matrix, id_to_index, df):
    """Récupère les top 5 voisins pour un produit ancre"""
    anchor_idx = id_to_index[anchor_id]
    sim_scores = similarity_matrix[anchor_idx]
    top_indices = np.argsort(sim_scores)[::-1][1:6]
    top_ids = [df.iloc[idx]['id'] for idx in top_indices]
    top_scores = [round(sim_scores[idx], 3) for idx in top_indices]
    return list(zip(top_ids, top_scores))

results_resnet = []
for anchor_id in ground_truth['anchor_id']:
    top5 = get_top5_neighbors(anchor_id, resnet_similarity, id_to_index, df)
    results_resnet.append({
        'anchor_id': anchor_id,
        'top5_ids': [x[0] for x in top5],
        'top5_scores': [x[1] for x in top5]
    })

df_resnet = pd.DataFrame(results_resnet)
print(f"Résultats ResNet-50 : {df_resnet.shape}")
print(df_resnet.head())

Résultats ResNet-50 : (30, 3)
   anchor_id                             top5_ids  \
0       3314    [4861, 41968, 8356, 40162, 34039]   
1       4543    [36464, 5739, 6852, 15743, 15270]   
2      15270   [17041, 14500, 4543, 36464, 49509]   
3      17993  [27906, 12677, 27018, 26822, 11279]   
4      20610  [35915, 36523, 38315, 36069, 53921]   

                           top5_scores  
0  [0.863, 0.862, 0.859, 0.859, 0.808]  
1   [0.938, 0.934, 0.93, 0.928, 0.926]  
2  [0.935, 0.932, 0.926, 0.918, 0.912]  
3  [0.937, 0.936, 0.921, 0.918, 0.917]  
4   [0.882, 0.87, 0.869, 0.852, 0.852]  


In [9]:
# Sauvegarder les résultats ResNet-50
np.save('results/resnet_similarity_matrix.npy', resnet_similarity)
np.save('results/resnet_embeddings.npy', resnet_embeddings)
df_resnet.to_csv('results/resnet_top5_anchors.csv', index=False)
print("ResNet-50 sauvegardé dans results/")

ResNet-50 sauvegardé dans results/


---
## Méthode 2 : Vision Transformer (ViT)

In [10]:
# Installer les packages nécessaires si besoin
# !pip install transformers

from transformers import ViTModel, ViTImageProcessor

# Charger le modèle ViT
model_name = 'google/vit-base-patch16-224'
vit_processor = ViTImageProcessor.from_pretrained(model_name)
vit_model = ViTModel.from_pretrained(model_name)
vit_model.eval()

print(f"ViT chargé : {model_name}")
print(f"Embedding CLS : 768 dimensions")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: aca8ae0c-43e7-4a9e-9d39-3eb50ab16ee2)')' thrown while requesting HEAD https://huggingface.co/google/vit-base-patch16-224/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f9fa8b6d-299c-4432-b319-262482a28057)')' thrown while requesting HEAD https://huggingface.co/google/vit-base-patch16-224/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f3736e0e-c220-47b6-a3d3-9d4d04f74deb)')' thrown while requesting HEAD https://huggingface.co/google/vit-base-patch16-224/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of ViTModel were not initialized from the model checkpoint at google/v

ViT chargé : google/vit-base-patch16-224
Embedding CLS : 768 dimensions


In [11]:
# Extraire les features pour toutes les images
vit_embeddings = []

print("Extraction des features ViT en cours...")
for idx, row in df.iterrows():
    img_name = row['image_path'].split('/')[-1]
    img_path = IMAGES_DIR / img_name
    
    if not img_path.exists():
        vit_embeddings.append(np.zeros(768))
        continue
    
    # Charger et prétraiter l'image
    img = Image.open(img_path).convert('RGB')
    inputs = vit_processor(images=img, return_tensors="pt")
    
    # Extraire l'embedding CLS
    with torch.no_grad():
        outputs = vit_model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()[0]
    
    vit_embeddings.append(cls_embedding)

vit_embeddings = np.array(vit_embeddings)
print(f"\nFeatures ViT : {vit_embeddings.shape}")

Extraction des features ViT en cours...

Features ViT : (400, 768)


In [12]:
# Matrice de similarité cosinus
vit_similarity = cosine_similarity(vit_embeddings)
print(f"Matrice de similarité : {vit_similarity.shape}")

Matrice de similarité : (400, 400)


In [13]:
# Extraction top-5 pour les ancres
results_vit = []
for anchor_id in ground_truth['anchor_id']:
    top5 = get_top5_neighbors(anchor_id, vit_similarity, id_to_index, df)
    results_vit.append({
        'anchor_id': anchor_id,
        'top5_ids': [x[0] for x in top5],
        'top5_scores': [x[1] for x in top5]
    })

df_vit = pd.DataFrame(results_vit)
print(f"Résultats ViT : {df_vit.shape}")
print(df_vit.head())

Résultats ViT : (30, 3)
   anchor_id                             top5_ids  \
0       3314    [8389, 4861, 39345, 34039, 26486]   
1       4543     [15270, 15743, 5739, 6852, 4445]   
2      15270   [17041, 14500, 6431, 34832, 15743]   
3      17993   [27906, 48391, 7885, 12677, 10018]   
4      20610  [58266, 53886, 21560, 21937, 36664]   

                           top5_scores  
0    [0.71, 0.647, 0.63, 0.628, 0.605]  
1   [0.696, 0.69, 0.688, 0.683, 0.683]  
2    [0.775, 0.76, 0.75, 0.722, 0.711]  
3     [0.676, 0.67, 0.653, 0.619, 0.6]  
4  [0.827, 0.782, 0.763, 0.763, 0.753]  


In [14]:
# Sauvegarder les résultats ViT
np.save('results/vit_similarity_matrix.npy', vit_similarity)
np.save('results/vit_embeddings.npy', vit_embeddings)
df_vit.to_csv('results/vit_top5_anchors.csv', index=False)
print("ViT sauvegardé dans results/")

ViT sauvegardé dans results/


---
## Analyse Comparative

In [15]:
# Comparaison pour une ancre exemple
test_anchor = ground_truth['anchor_id'].iloc[0]

print(f"=== Ancre {test_anchor} ===")
anchor_info = df[df['id'] == test_anchor].iloc[0]
print(f"Produit : {anchor_info['nom']}")
print(f"Catégorie : {anchor_info['categorie']}")
print()

# Ground truth (visual)
gt_row = ground_truth[ground_truth['anchor_id'] == test_anchor].iloc[0]
print("Ground Truth (visual) :")
for i in range(1, 6):
    prod_id = gt_row[f'visual_id{i}']
    prod_info = df[df['id'] == prod_id].iloc[0]
    print(f"  {i}. {prod_id} - {prod_info['nom']} [{prod_info['categorie']}]")
print()

# ResNet-50
row_resnet = df_resnet[df_resnet['anchor_id'] == test_anchor].iloc[0]
print("ResNet-50 :")
for i, (prod_id, score) in enumerate(zip(row_resnet['top5_ids'], row_resnet['top5_scores']), 1):
    prod_info = df[df['id'] == prod_id].iloc[0]
    print(f"  {i}. {prod_id} - {prod_info['nom']} [{prod_info['categorie']}] ({score})")
print()

# ViT
row_vit = df_vit[df_vit['anchor_id'] == test_anchor].iloc[0]
print("ViT :")
for i, (prod_id, score) in enumerate(zip(row_vit['top5_ids'], row_vit['top5_scores']), 1):
    prod_info = df[df['id'] == prod_id].iloc[0]
    print(f"  {i}. {prod_id} - {prod_info['nom']} [{prod_info['categorie']}] ({score})")

=== Ancre 3314 ===
Produit : Guerrilla Men's Warrior Brown T-shirt
Catégorie : Topwear

Ground Truth (visual) :
  1. 4861 - Nike Men Arsenal Yellow Jersey [Topwear]
  2. 26486 - Puma Men Large Logo Ringer Blue T-shirt [Topwear]
  3. 31279 - Puma Men Bonn Graphic Navy Blue T-shirt [Topwear]
  4. 24146 - ADIDAS Men Black T-shirt [Topwear]
  5. 38808 - Music Men Blue Printed T-shirt [Topwear]

ResNet-50 :
  1. 4861 - Nike Men Arsenal Yellow Jersey [Topwear] (0.8629999756813049)
  2. 41968 - Gini and Jony Boys United White T-shirt [Topwear] (0.8619999885559082)
  3. 8356 - Doodle Boy's End Is Just Begining Black Kidswear [Topwear] (0.859000027179718)
  4. 40162 - Gini and Jony Boys Printed Yellow T-Shirt [Topwear] (0.859000027179718)
  5. 34039 - Palm Tree Boys Check Red Shirt [Topwear] (0.8080000281333923)

ViT :
  1. 8389 - ADIDAS Men  Red Polo Tshirts [Topwear] (0.7099999785423279)
  2. 4861 - Nike Men Arsenal Yellow Jersey [Topwear] (0.6470000147819519)
  3. 39345 - Mr. Men Blue T-shir

In [16]:
# Comparaison globale avec ground truth (visual)
def calculate_match_rate(predictions, ground_truth_col):
    """Calcule le taux de correspondance avec ground truth"""
    matches = 0
    total = 0
    for _, row in ground_truth.iterrows():
        anchor_id = row['anchor_id']
        gt_ids = [row[f'{ground_truth_col}_id{i}'] for i in range(1, 6)]
        
        pred_row = predictions[predictions['anchor_id'] == anchor_id].iloc[0]
        pred_ids = pred_row['top5_ids']
        
        matches += len(set(pred_ids) & set(gt_ids))
        total += 5
    
    return matches / total

resnet_match = calculate_match_rate(df_resnet, 'visual')
vit_match = calculate_match_rate(df_vit, 'visual')

print("=== Comparaison avec Ground Truth (visual) ===")
print(f"ResNet-50 : {resnet_match:.1%} de correspondance")
print(f"ViT       : {vit_match:.1%} de correspondance")
print()
if vit_match > resnet_match:
    print("→ ViT est plus proche des jugements humains")
else:
    print("→ ResNet-50 est plus proche des jugements humains")

=== Comparaison avec Ground Truth (visual) ===
ResNet-50 : 44.0% de correspondance
ViT       : 48.0% de correspondance

→ ViT est plus proche des jugements humains


In [17]:
# Identifier voisins visuels ≠ sémantiques
def find_visual_not_semantic(df_viz, df_sem):
    """Trouve les produits visuellement similaires mais sémantiquement différents"""
    examples = []
    
    for _, row in ground_truth.iterrows():
        anchor_id = row['anchor_id']
        
        # Top 5 visuel (ResNet ou ViT)
        viz_row = df_viz[df_viz['anchor_id'] == anchor_id].iloc[0]
        viz_ids = set(viz_row['top5_ids'])
        
        # Top 5 sémantique (TF-IDF)
        sem_row = df_sem[df_sem['anchor_id'] == anchor_id].iloc[0]
        sem_ids = set(sem_row['top5_ids'])
        
        # Info ancre
        anchor_info = df[df['id'] == anchor_id].iloc[0]
        anchor_cat = anchor_info['categorie']
        
        # Produits visuels mais PAS sémantiques ET catégories différentes
        visual_only = viz_ids - sem_ids
        
        for prod_id in visual_only:
            # Vérifier catégorie différente
            prod_info = df[df['id'] == prod_id].iloc[0]
            if prod_info['categorie'] != anchor_cat:
                examples.append({
                    'anchor': f"{anchor_id} - {anchor_info['nom']}",
                    'anchor_cat': anchor_cat,
                    'product': f"{prod_id} - {prod_info['nom']}",
                    'product_cat': prod_info['categorie']
                })
        
        # Max 2 exemples par ancre pour garder lisible
        if len(examples) > 20:
            break
    
    return pd.DataFrame(examples)

# Charger les résultats sémantiques (TF-IDF)
df_tfidf = pd.read_csv('results/tfidf_top5_anchors.csv')

# Trouver les exemples avec ResNet
visual_not_semantic = find_visual_not_semantic(df_resnet, df_tfidf)

print("=== Voisins Visuels ≠ Sémantiques ===")
print("Produits qui se ressemblent visuellement mais ont des catégories différentes :\n")
for idx, row in visual_not_semantic.head(15).iterrows():
    print(f"Ancre  : {row['anchor']} [{row['anchor_cat']}]")
    print(f"Produit: {row['product']} [{row['product_cat']}]")
    print()

=== Voisins Visuels ≠ Sémantiques ===
Produits qui se ressemblent visuellement mais ont des catégories différentes :

Ancre  : 24532 - Mother Earth Women Off white Kurta [Topwear]
Produit: 42805 - Little Miss Intimates Women Yellow Brief [Innerwear]

Ancre  : 33057 - U.S. Polo Assn. Denim Co. Men Black Shirt [Topwear]
Produit: 25394 - Levis Men White Innerwear T-shirt [Innerwear]

Ancre  : 42804 - Little Miss Intimates Women Grey Brief [Innerwear]
Produit: 32579 - ONLY Women Turquoise Chiffon Top [Topwear]

Ancre  : 42804 - Little Miss Intimates Women Grey Brief [Innerwear]
Produit: 24532 - Mother Earth Women Off white Kurta [Topwear]

Ancre  : 42804 - Little Miss Intimates Women Grey Brief [Innerwear]
Produit: 42200 - Sepia Women Yellow Top [Topwear]

Ancre  : 48348 - Pitaraa Women Green Clutch [Bags]
Produit: 50526 - Chromozome Men Pack of 2 Briefs [Innerwear]

